In [1]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class CandleDataset(Dataset):
    def __init__(self, df, feature_cols, window=60):
        self.features = df[feature_cols].values.astype(np.float32)
        self.labels = df["label"].values.astype(np.int64)
        self.window = window

    def __len__(self):
        return len(self.features) - self.window

    def __getitem__(self, idx):
        x = self.features[idx : idx + self.window]          # (window, n_features)
        y = self.labels[idx + self.window - 1]                # лейбл на конец окна
        return torch.tensor(x), torch.tensor(y)

In [2]:
import torch

print(torch.__version__)  
print(torch.cuda.is_available())

2.13.0+cu132
True


In [3]:
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score
from model import LSTMClassifier
import torch.nn as nn

def train_model(model, train_loader, val_loader, epochs=30, lr=1e-3, device="cuda"):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # валидация
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device)
                logits = model(x)
                preds = logits.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(y.numpy())

        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average="macro")
        print(f"Epoch {epoch+1}: train_loss={train_loss/len(train_loader):.4f}, val_acc={acc:.4f}, val_f1={f1:.4f}")

    return model

In [ ]:
import pandas as pd
from model import prepare_features_and_labels, feature_cols

# load train data
df = pd.read_parquet("data/synthetic/train/mins_518400_seed1047423_bodysigma0046_wickdf23.parquet")
df = prepare_features_and_labels(df, horizon=5, flat_threshold=0.001)

# train/val split
split_idx = int(len(df) * 0.8)
train_df, val_df = df.iloc[:split_idx], df.iloc[split_idx:]

train_ds = CandleDataset(train_df, feature_cols, window=60)
val_ds = CandleDataset(val_df, feature_cols, window=60)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)   # shuffle only batches
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

model = LSTMClassifier(n_features=len(feature_cols), num_classes=3)
print(f"Learning on {'cuda' if torch.cuda.is_available() else 'cpu'}")
model = train_model(model, train_loader, val_loader, epochs=60, device="cuda" if torch.cuda.is_available() else "cpu")

timestamp          0
open               0
high               0
low                0
close              0
volume             0
quote_volume       0
return             1
high_low_range    59
body              59
volatility_20     20
label              0
dtype: int64
Before dropna: 1440
After dropna: 1381
Learning on cuda
Epoch 1: train_loss=1.0674, val_acc=0.5161, val_f1=0.2270
Epoch 2: train_loss=0.8280, val_acc=0.5161, val_f1=0.2270
Epoch 3: train_loss=0.7722, val_acc=0.5161, val_f1=0.2270
Epoch 4: train_loss=0.7571, val_acc=0.5161, val_f1=0.2270
Epoch 5: train_loss=0.7425, val_acc=0.5161, val_f1=0.2270
Epoch 6: train_loss=0.7359, val_acc=0.5161, val_f1=0.2270
Epoch 7: train_loss=0.7368, val_acc=0.5161, val_f1=0.2270
Epoch 8: train_loss=0.7356, val_acc=0.5161, val_f1=0.2270
Epoch 9: train_loss=0.7353, val_acc=0.5161, val_f1=0.2270
Epoch 10: train_loss=0.7357, val_acc=0.5161, val_f1=0.2270
Epoch 11: train_loss=0.7333, val_acc=0.5161, val_f1=0.2270
Epoch 12: train_loss=0.7285, val_acc=0.

In [ ]:
torch.save(model.state_dict(), "models/lstm_518400_60_0046_23.pt")